# Fire Detection System (YOLOv5 + Gradio)

This notebook trains a fire detection model from scratch using the `ultralytics` library and provides a Gradio interface for testing on images and videos.

**Features:**
- Automatic GPU detection (Colab/Local).
- Dynamic dataset configuration.
- Interactive Web UI.

In [ ]:
# 1. Install Dependencies
%pip install ultralytics gradio

In [5]:
# 2. Setup Environment & Paths
import os
import torch
from ultralytics import YOLO

# Check for GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

is_colab = os.path.exists('/content')

# Define Dataset Path Dynamically
if is_colab:
    print("Running on Google Colab")
    
    # --- COLAB DATA SETUP ---
    # Option A: Mount Google Drive (Recommended if dataset is in Drive)
    from google.colab import drive
    drive.mount('/content/drive')
    
    # IMPORTANT: Update this path to where your 'datasets' folder is in your Drive
    # Example: '/content/drive/MyDrive/fire_detection/datasets/fire/fire'
    # We default to a standard upload location if not found
    start_path = '/content/datasets/fire/fire' 
    
    # Check user drive path if it exists (Adjust 'MyDrive/...' as needed)
    possible_drive_path = '/content/drive/MyDrive/datasets/fire/fire'
    if os.path.exists(possible_drive_path):
        start_path = possible_drive_path
        
    if not os.path.exists(start_path):
        print(f"\n⚠️ DATASET NOT FOUND AT: {start_path}")
        print("Please either:")
        print("1. Upload your 'datasets' folder to your Google Drive and update the path above.")
        print("2. Or zip your 'datasets' folder, upload it to the Files tab on the left, and unzip it.")
        print("   !unzip datasets.zip -d /content/")
else:
    print("Running Locally")
    # If running locally in 'results/', dataset is at '../datasets/fire/fire'
    start_path = os.path.abspath(os.path.join(os.getcwd(), '../datasets/fire/fire'))

print(f"Dataset root set to: {start_path}")

Using device: cuda
GPU: Tesla T4
Running on Google Colab


KeyboardInterrupt: 

In [ ]:
# 3. Create Custom Configuration File (custom_fire.yaml)
import yaml

config = {
    'path': start_path,  # Root directory
    'train': 'train/images',
    'val': 'val/images',
    'nc': 1,
    'names': ['fire']
}

yaml_path = 'custom_fire.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(config, f)

print(f"Configuration saved to {yaml_path}")

# Verify data exists before training to avoid cryptic errors
train_path = os.path.join(start_path, config['train'])
if not os.path.exists(train_path):
    raise FileNotFoundError(f"Train images not found at {train_path}. Please check your dataset upload.")

In [ ]:
# 4. Train YOLOv5 Model
model = YOLO('yolov5su.pt')

print("Starting Training...")
results = model.train(
    data=yaml_path,
    epochs=10,        
    imgsz=640,
    device=0 if device == 'cuda' else 'cpu',
    project='projects/fire_detection',
    name='train_run',
    exist_ok=True
)

In [ ]:
# 5. Gradio Interface (Images & Video)
import gradio as gr
import cv2
import numpy as np
from PIL import Image

# Load the best trained model
best_model_path = 'projects/fire_detection/train_run/weights/best.pt'

# Fallback check
if os.path.exists(best_model_path):
    print(f"Loading trained model from {best_model_path}")
    inference_model = YOLO(best_model_path)
else:
    print("Training output not found (did training complete?), loading base model for demo.")
    inference_model = YOLO('yolov5su.pt')

def detect_image(image):
    # Run inference
    results = inference_model(image)
    # Plot results on the image
    res_plotted = results[0].plot()
    return res_plotted

def detect_video(video_path):
    # Process video frame by frame
    cap = cv2.VideoCapture(video_path)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    
    output_path = "output_detected.mp4"
    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
            
        # Detect
        results = inference_model(frame)
        annotated_frame = results[0].plot()
        
        out.write(annotated_frame)
        
    cap.release()
    out.release()
    return output_path

# Build UI
with gr.Blocks() as demo:
    gr.Markdown("# 🔥 Fire Detection System")
    
    with gr.Tab("Image Detection"):
        image_input = gr.Image(type="pil", label="Upload Image")
        image_output = gr.Image(label="Detected Fire")
        image_btn = gr.Button("Detect")
        image_btn.click(detect_image, inputs=image_input, outputs=image_output)
        
    with gr.Tab("Video Detection"):
        video_input = gr.Video(label="Upload Video")
        video_output = gr.Video(label="Processed Video")
        video_btn = gr.Button("Detect")
        video_btn.click(detect_video, inputs=video_input, outputs=video_output)

demo.launch(share=True)